# PHASE 4: Feature Engineering & Labeling
**Traceability**
- Issue ID: #4 Feature Engineering & Labeling

**External References**:
- [Wassim Derbel - NASA Predictive Maintenance](https://www.kaggle.com/code/wassimderbel/nasa-predictive-maintenance-rul) (Piecewise RUL, Rolling Features)

## 1. Objectives
- **Reasoning**: Raw RUL is linear, but degradation is non-linear. We clip RUL to reflect the "healthy" state.
- **Reasoning**: Sensor data is noisy. Rolling means/stds capture the underlying trend better than raw values.
- **Action**: Generate Piecewise RUL and Rolling Window Features.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ── Plotting Config ───────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11
})
COLORS = ['#1F4E79', '#2E75B6', '#70AD47', '#FF7043', '#AB47BC']

# ── Global Config ────────────────────────────────────────────────────────
PROCESSED_DIR = Path('../data/processed')
DATA_DIR = Path('../data')
CLIP_VALUE = 125  # Optimum based on FD001 literature
WINDOW = 10       # Standard window size for rolling features
W1 = 30           # Warning threshold (binary class)
W0 = 15           # Critical threshold (multi-class)

# Load Data
df_train = pd.read_csv(PROCESSED_DIR / 'train_cleaned.csv')
df_test = pd.read_csv(PROCESSED_DIR / 'test_cleaned.csv')
sensor_cols = [c for c in df_train.columns if c.startswith('s_')]
print(f"✅ Data loaded: Train {df_train.shape}, Test {df_test.shape}")

### 4.1 RUL Label Engineering (Piecewise Clipping)
We calculate the Remaining Useful Life (RUL) for each row. 

**Reasoning**: In the early stages of an engine's life, there is no degradation. Assuming a linear decline from cycle 0 introduces error. We clip the RUL at 125 cycles to model this "healthy" plateau.

In [ ]:
def add_rul_labels(df_train, df_test):
    """Compute and clip RUL for training and test data."""
    # 1. Train RUL
    rul_df = df_train.groupby('unit_number')['time_cycles'].max().reset_index()
    rul_df.columns = ['unit_number', 'max_cycle']
    df_train = df_train.merge(rul_df, on='unit_number')
    df_train['RUL_raw'] = df_train['max_cycle'] - df_train['time_cycles']
    df_train['RUL'] = df_train['RUL_raw'].clip(upper=CLIP_VALUE)
    df_train.drop(columns='max_cycle', inplace=True)
    
    # 2. Test RUL (backfilled from ground truth)
    y_test_rul = pd.read_csv(DATA_DIR / 'RUL_FD001.txt', sep=r'\s+', header=None, index_col=False, names=['RUL'])
    last_cycle = df_test.groupby('unit_number')['time_cycles'].max().reset_index()
    last_cycle.columns = ['unit_number', 'last_cycle']
    last_cycle['rul_at_end'] = y_test_rul['RUL'].values
    
    df_test = df_test.merge(last_cycle, on='unit_number')
    df_test['RUL_raw'] = df_test['rul_at_end'] + (df_test['last_cycle'] - df_test['time_cycles'])
    df_test['RUL'] = df_test['RUL_raw'].clip(upper=CLIP_VALUE)
    df_test.drop(columns=['last_cycle', 'rul_at_end'], inplace=True)
    
    return df_train, df_test

df_train, df_test = add_rul_labels(df_train, df_test)
print(f"✅ RUL labels engineered (Clipped at {CLIP_VALUE}).")

# Diagnostic: Visualize RUL
def visualize_piecewise_rul(df, unit=1):
    """Visualize raw vs. clipped RUL for a specific engine."""
    subset = df[df['unit_number'] == unit]
    
    plt.figure(figsize=(10, 5))
    plt.plot(subset['time_cycles'], subset['RUL_raw'], label='Raw RUL', color=COLORS[3], linestyle='--')
    plt.plot(subset['time_cycles'], subset['RUL'], label=f'Piecewise RUL (Clip={CLIP_VALUE})', color=COLORS[0], linewidth=2)
    plt.axhline(y=CLIP_VALUE, color='gray', linestyle=':', alpha=0.5)
    plt.title(f'Piecewise RUL Labeling (Engine {unit})')
    plt.xlabel('Time Cycles')
    plt.ylabel('RUL')
    plt.legend()
    plt.show()

visualize_piecewise_rul(df_train, unit=1)

### 4.2 Feature Engineering: Rolling Statistics
We calculate rolling means and standard deviations to smooth out sensor noise.

**Reasoning**: Raw sensor readings can fluctuate wildly. A rolling window (size=10) extracts the signal from the noise.

In [ ]:
def add_rolling_features(df, sensors, window=10):
    """Add rolling stats per sensor column (leakage-safe)."""
    df = df.copy().sort_values(['unit_number', 'time_cycles'])
    for col in sensors:
        grouped = df.groupby('unit_number')[col]
        df[f'{col}_rm']   = grouped.transform(lambda x: x.rolling(window, min_periods=1).mean())
        df[f'{col}_rstd'] = grouped.transform(lambda x: x.rolling(window, min_periods=1).std().fillna(0))
    return df

df_train = add_rolling_features(df_train, sensor_cols, window=WINDOW)
df_test = add_rolling_features(df_test, sensor_cols, window=WINDOW)
print(f"✅ Rolling window features added.")

# Diagnostic: Rolling Effect Visualization
def visualize_rolling_effect(df, sensor='s_11', unit=1):
    """Visualize raw sensor vs. rolling mean."""
    subset = df[df['unit_number'] == unit]
    
    plt.figure(figsize=(12, 5))
    plt.plot(subset['time_cycles'], subset[sensor], label='Raw Sensor', color='gray', alpha=0.4)
    plt.plot(subset['time_cycles'], subset[f'{sensor}_rm'], label=f'Rolling Mean (w={WINDOW})', color=COLORS[1], linewidth=2)
    plt.title(f'Rolling Window Smoothing: {sensor} (Engine {unit})')
    plt.xlabel('Time Cycles')
    plt.ylabel('Value')
    plt.legend()
    plt.show()

visualize_rolling_effect(df_train, sensor='s_11', unit=1)

### 4.3 Classification Labels & Domain Features
Adding binary labels for classification tasks and cycle normalization.

In [ ]:
def add_classification_labels(df):
    """Add binary and 3-class classification labels."""
    df['label1'] = (df['RUL_raw'] <= W1).astype(int)
    df['label2'] = df['label1'].copy()
    df.loc[df['RUL_raw'] <= W0, 'label2'] = 2
    return df

def add_domain_features(df):
    """Add cycle normalization and sensor differencing."""
    df = df.copy()
    max_c = df.groupby('unit_number')['time_cycles'].transform('max')
    df['cycle_norm'] = df['time_cycles'] / max_c
    return df

df_train = add_classification_labels(df_train)
df_test = add_classification_labels(df_test)
df_train = add_domain_features(df_train)
df_test = add_domain_features(df_test)

# Save
df_train.to_csv(PROCESSED_DIR / 'train_labeled.csv', index=False)
df_test.to_csv(PROCESSED_DIR / 'test_labeled.csv', index=False)
print(f"\n✅ Labeled & engineered data saved to {PROCESSED_DIR}")